# Machine Learning Course Methods — Companion Notebook

**Purpose:** a compact, runnable reference for the machine-learning methods covered in the course.  
This notebook is **separate from the Student Risk OULAD project notebook**. Its purpose is syllabus coverage and viva revision, not final-model selection for the project.

## Course topics represented here
- Supervised learning: classification and regression
- Logistic Regression, Naive Bayes, KNN, SVM, Decision Tree
- Linear Regression
- KNN / SVM / Decision Tree in regression where applicable
- Train/test split, missing-value handling, encoding, scaling
- Accuracy, Precision, Recall, F1, ROC-AUC, Confusion Matrix
- MSE and Log Loss
- 5-Fold Cross Validation and Hyperparameter Tuning
- K-Means, K-Means++, DBSCAN, GMM
- PCA
- Reinforcement Learning
- Deep Learning concepts: neural networks, ANN, activation functions, optimizers, classification/regression, CNN

> **Terminology note:** the course notes mention `KNN++`, but no standard algorithm definition was supplied with that name. This notebook implements standard **KNN** and demonstrates **K-Means++** separately rather than inventing an undefined method.


## 1. Setup


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import make_classification, make_regression, make_blobs, load_digits
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    confusion_matrix, log_loss, mean_squared_error, mean_absolute_error,
    r2_score, silhouette_score
)
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.svm import SVC, SVR
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.ensemble import RandomForestClassifier
from sklearn.cluster import KMeans, DBSCAN
from sklearn.mixture import GaussianMixture
from sklearn.decomposition import PCA
from sklearn.neural_network import MLPClassifier, MLPRegressor

RANDOM_STATE = 42
print("Ready.")


## 2. Core ML workflow

A typical supervised-learning workflow is:

**Define target → split data → preprocess → train → evaluate → cross-validate → tune → select**

For classification, the target is categorical.  
For regression, the target is continuous.


## 3. Preprocessing: missing values, encoding, scaling, train/test split


In [ ]:
X_num, y_cls = make_classification(
    n_samples=1200, n_features=6, n_informative=4, n_redundant=1,
    weights=[0.58, 0.42], class_sep=1.1, random_state=RANDOM_STATE
)

X_cls = pd.DataFrame(X_num, columns=[f"num_{i}" for i in range(6)])
X_cls["study_mode"] = np.where(X_cls["num_0"] > X_cls["num_0"].median(), "online", "blended")
X_cls["support_level"] = pd.cut(
    X_cls["num_1"], bins=[-np.inf, -0.5, 0.5, np.inf],
    labels=["low", "medium", "high"]
).astype(str)

rng = np.random.default_rng(RANDOM_STATE)
for col in ["num_2", "num_4", "study_mode"]:
    idx = rng.choice(len(X_cls), size=35, replace=False)
    X_cls.loc[idx, col] = np.nan

X_train_cls, X_test_cls, y_train_cls, y_test_cls = train_test_split(
    X_cls, y_cls, test_size=0.2, stratify=y_cls, random_state=RANDOM_STATE
)

numeric_cols = [c for c in X_cls.columns if c.startswith("num_")]
categorical_cols = ["study_mode", "support_level"]

numeric_scaled = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

preprocess_scaled = ColumnTransformer([
    ("num", numeric_scaled, numeric_cols),
    ("cat", categorical_pipe, categorical_cols)
])

preprocess_unscaled = ColumnTransformer([
    ("num", SimpleImputer(strategy="median"), numeric_cols),
    ("cat", categorical_pipe, categorical_cols)
])

print("Train rows:", len(X_train_cls))
print("Test rows :", len(X_test_cls))
print("Missing values handled inside pipelines.")


### Why preprocessing differs by model
- **Scaling is important** for distance/margin-based models such as KNN and SVM.
- Logistic Regression also commonly benefits from scaled numeric inputs.
- Decision Trees do not require scaling.
- Categorical data is encoded numerically before entering scikit-learn estimators.


## 4. Supervised Classification Models


In [ ]:
classification_models = {
    "Logistic Regression": Pipeline([
        ("prep", preprocess_scaled),
        ("model", LogisticRegression(max_iter=2000, random_state=RANDOM_STATE))
    ]),
    "Naive Bayes": Pipeline([
        ("prep", preprocess_scaled),
        ("model", GaussianNB())
    ]),
    "KNN": Pipeline([
        ("prep", preprocess_scaled),
        ("model", KNeighborsClassifier(n_neighbors=7))
    ]),
    "SVM": Pipeline([
        ("prep", preprocess_scaled),
        ("model", SVC(kernel="rbf", probability=True, random_state=RANDOM_STATE))
    ]),
    "Decision Tree": Pipeline([
        ("prep", preprocess_unscaled),
        ("model", DecisionTreeClassifier(max_depth=6, random_state=RANDOM_STATE))
    ]),
    "Random Forest (extension)": Pipeline([
        ("prep", preprocess_unscaled),
        ("model", RandomForestClassifier(n_estimators=120, random_state=RANDOM_STATE))
    ]),
}

rows = []
fitted_classifiers = {}

for name, model in classification_models.items():
    model.fit(X_train_cls, y_train_cls)
    pred = model.predict(X_test_cls)
    prob = model.predict_proba(X_test_cls)[:, 1]
    fitted_classifiers[name] = model
    rows.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test_cls, pred),
        "Precision": precision_score(y_test_cls, pred),
        "Recall": recall_score(y_test_cls, pred),
        "F1": f1_score(y_test_cls, pred),
        "ROC-AUC": roc_auc_score(y_test_cls, prob),
        "Log Loss": log_loss(y_test_cls, prob),
    })

classification_results = pd.DataFrame(rows).sort_values("F1", ascending=False)
classification_results.round(4)


### Classification metrics
- **Accuracy:** overall proportion predicted correctly.
- **Precision:** among predicted positives, how many are truly positive.
- **Recall:** among actual positives, how many were found.
- **F1:** harmonic mean of Precision and Recall.
- **ROC-AUC:** ranking quality across classification thresholds.
- **Log Loss:** evaluates predicted probabilities; lower is better.


### Confusion Matrix


In [ ]:
best_cls_name = classification_results.iloc[0]["Model"]
best_cls = fitted_classifiers[best_cls_name]
best_pred = best_cls.predict(X_test_cls)
cm = confusion_matrix(y_test_cls, best_pred)

print("Best by test-set F1 in this teaching demo:", best_cls_name)
display(pd.DataFrame(
    cm,
    index=["Actual 0", "Actual 1"],
    columns=["Predicted 0", "Predicted 1"]
))

plt.figure(figsize=(4.5, 4))
plt.imshow(cm)
plt.title(f"Confusion Matrix — {best_cls_name}")
plt.xlabel("Predicted class")
plt.ylabel("Actual class")
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        plt.text(j, i, cm[i, j], ha="center", va="center")
plt.tight_layout()
plt.show()


## 5. 5-Fold Cross Validation


In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc",
}

cv_rows = []
for name, model in classification_models.items():
    scores = cross_validate(model, X_cls, y_cls, cv=cv, scoring=scoring, n_jobs=-1)
    cv_rows.append({
        "Model": name,
        "Accuracy": scores["test_accuracy"].mean(),
        "Precision": scores["test_precision"].mean(),
        "Recall": scores["test_recall"].mean(),
        "F1": scores["test_f1"].mean(),
        "ROC-AUC": scores["test_roc_auc"].mean(),
    })

cv_results = pd.DataFrame(cv_rows).sort_values("F1", ascending=False)
cv_results.round(4)


Cross-validation estimates how consistently a model performs across multiple splits instead of relying on one train/test split.


## 6. Hyperparameter Tuning


In [ ]:
knn_pipeline = Pipeline([
    ("prep", preprocess_scaled),
    ("model", KNeighborsClassifier())
])

param_grid = {
    "model__n_neighbors": [3, 5, 7, 11],
    "model__weights": ["uniform", "distance"],
}

grid = GridSearchCV(
    knn_pipeline,
    param_grid=param_grid,
    cv=5,
    scoring="f1",
    n_jobs=-1
)
grid.fit(X_train_cls, y_train_cls)

print("Best KNN parameters:", grid.best_params_)
print("Best CV F1:", round(grid.best_score_, 4))


## 7. Supervised Regression

Regression predicts a **continuous numeric target**.  
The course methods demonstrated here are:
- Linear Regression
- KNN Regressor
- Support Vector Regression (SVR)
- Decision Tree Regressor


In [ ]:
X_reg, y_reg = make_regression(
    n_samples=1000, n_features=8, n_informative=6,
    noise=18, random_state=RANDOM_STATE
)

X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=RANDOM_STATE
)

regression_models = {
    "Linear Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LinearRegression())
    ]),
    "KNN Regressor": Pipeline([
        ("scaler", StandardScaler()),
        ("model", KNeighborsRegressor(n_neighbors=7))
    ]),
    "SVR": Pipeline([
        ("scaler", StandardScaler()),
        ("model", SVR(C=10, epsilon=0.1))
    ]),
    "Decision Tree Regressor": DecisionTreeRegressor(
        max_depth=8, random_state=RANDOM_STATE
    ),
}

reg_rows = []
for name, model in regression_models.items():
    model.fit(X_train_reg, y_train_reg)
    pred = model.predict(X_test_reg)
    mse = mean_squared_error(y_test_reg, pred)
    reg_rows.append({
        "Model": name,
        "MSE": mse,
        "RMSE": np.sqrt(mse),
        "MAE": mean_absolute_error(y_test_reg, pred),
        "R2": r2_score(y_test_reg, pred),
    })

regression_results = pd.DataFrame(reg_rows).sort_values("RMSE")
regression_results.round(4)


### Regression metrics
- **MSE:** mean squared error; lower is better.
- **RMSE:** square root of MSE, in the target's original units.
- **MAE:** average absolute error.
- **R²:** proportion of variance explained by the model.


## 8. K-Means and K-Means++ Clustering


In [ ]:
X_blob, y_blob_true = make_blobs(
    n_samples=700, centers=4, cluster_std=1.0, random_state=RANDOM_STATE
)

kmeans_random = KMeans(
    n_clusters=4, init="random", n_init=10, random_state=RANDOM_STATE
)
kmeans_plus = KMeans(
    n_clusters=4, init="k-means++", n_init=10, random_state=RANDOM_STATE
)

labels_random = kmeans_random.fit_predict(X_blob)
labels_plus = kmeans_plus.fit_predict(X_blob)

kmeans_compare = pd.DataFrame([
    {
        "Method": "K-Means (random initialization)",
        "Inertia": kmeans_random.inertia_,
        "Silhouette": silhouette_score(X_blob, labels_random),
    },
    {
        "Method": "K-Means++ initialization",
        "Inertia": kmeans_plus.inertia_,
        "Silhouette": silhouette_score(X_blob, labels_plus),
    },
])

kmeans_compare.round(4)


In [ ]:
plt.figure(figsize=(6, 4))
plt.scatter(X_blob[:, 0], X_blob[:, 1], c=labels_plus, s=15)
plt.title("K-Means++ clustering")
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.tight_layout()
plt.show()


**K-Means++** is an improved centroid-initialization strategy for K-Means. It chooses starting centroids more carefully than purely random initialization.


## 9. DBSCAN


In [ ]:
dbscan = DBSCAN(eps=0.5, min_samples=5)
db_labels = dbscan.fit_predict(X_blob)

n_clusters = len(set(db_labels)) - (1 if -1 in db_labels else 0)
n_noise = int(np.sum(db_labels == -1))

print("Estimated clusters:", n_clusters)
print("Noise points:", n_noise)

plt.figure(figsize=(6, 4))
plt.scatter(X_blob[:, 0], X_blob[:, 1], c=db_labels, s=15)
plt.title("DBSCAN clustering")
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.tight_layout()
plt.show()


DBSCAN is density-based. Unlike K-Means, it can mark points as noise and does not require specifying the exact number of clusters in advance.


## 10. Gaussian Mixture Model (GMM)


In [ ]:
gmm = GaussianMixture(n_components=4, random_state=RANDOM_STATE)
gmm_labels = gmm.fit_predict(X_blob)
gmm_prob = gmm.predict_proba(X_blob)

print("GMM cluster counts:")
display(pd.Series(gmm_labels).value_counts().sort_index().rename("count").to_frame())
print("Example soft-membership probabilities:")
display(pd.DataFrame(gmm_prob[:5]).round(3))


GMM is probabilistic clustering. Each sample can have a probability of belonging to each component instead of only a hard cluster label.


## 11. PCA — Principal Component Analysis


In [ ]:
digits = load_digits()
X_digits = digits.data
y_digits = digits.target

X_digits_scaled = StandardScaler().fit_transform(X_digits)
pca = PCA(n_components=2, random_state=RANDOM_STATE)
X_pca = pca.fit_transform(X_digits_scaled)

print("Original dimensions:", X_digits.shape[1])
print("Reduced dimensions:", X_pca.shape[1])
print("Explained variance ratio:", np.round(pca.explained_variance_ratio_, 4))
print("Total variance retained by 2 PCs:", round(pca.explained_variance_ratio_.sum(), 4))

plt.figure(figsize=(6, 4))
plt.scatter(X_pca[:, 0], X_pca[:, 1], c=y_digits, s=12)
plt.title("Digits projected onto first two principal components")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.tight_layout()
plt.show()


PCA is a **dimensionality-reduction** technique. It creates new orthogonal components that capture as much variance as possible.


## 12. Reinforcement Learning — Simple Q-Learning Example


In [ ]:
# A minimal 1-D environment:
# states 0..5, goal at state 5. Actions: 0=left, 1=right.
n_states = 6
n_actions = 2
goal_state = 5

Q = np.zeros((n_states, n_actions))
alpha = 0.2
gamma = 0.95
epsilon = 0.2
episodes = 600
rng = np.random.default_rng(RANDOM_STATE)

for _ in range(episodes):
    state = 0
    for step in range(30):
        if rng.random() < epsilon:
            action = rng.integers(n_actions)
        else:
            action = int(np.argmax(Q[state]))

        next_state = max(0, state - 1) if action == 0 else min(goal_state, state + 1)
        reward = 10 if next_state == goal_state else -0.1

        Q[state, action] += alpha * (
            reward + gamma * np.max(Q[next_state]) - Q[state, action]
        )

        state = next_state
        if state == goal_state:
            break

policy = np.argmax(Q, axis=1)
display(pd.DataFrame(Q, columns=["Left", "Right"]).round(3))
print("Learned action per state (0=Left, 1=Right):", policy.tolist())


### RL idea
Reinforcement Learning learns from **rewards and interaction**, not from a fixed labeled target in the same way as supervised learning.  
The Q-Learning example above learns that moving toward the goal state produces the best long-term reward.


## 13. Deep Learning Foundations — Activation Functions


In [ ]:
x = np.linspace(-5, 5, 300)
sigmoid = 1 / (1 + np.exp(-x))
tanh = np.tanh(x)
relu = np.maximum(0, x)

plt.figure(figsize=(7, 4))
plt.plot(x, sigmoid, label="Sigmoid")
plt.plot(x, tanh, label="Tanh")
plt.plot(x, relu, label="ReLU")
plt.title("Common activation functions")
plt.xlabel("x")
plt.ylabel("activation(x)")
plt.legend()
plt.tight_layout()
plt.show()


Common activation functions:
- **Sigmoid:** output between 0 and 1; common for binary-classification output.
- **Tanh:** output between -1 and 1.
- **ReLU:** widely used in hidden layers because of its simple nonlinearity.

Typical optimizers include **SGD**, **Momentum**, **RMSprop**, and **Adam**. Optimizers update network weights to reduce the loss function.


## 14. ANN — Classification


In [ ]:
ann_cls = Pipeline([
    ("prep", preprocess_scaled),
    ("model", MLPClassifier(
        hidden_layer_sizes=(32, 16),
        activation="relu",
        solver="adam",
        max_iter=500,
        random_state=RANDOM_STATE
    ))
])

ann_cls.fit(X_train_cls, y_train_cls)
ann_pred = ann_cls.predict(X_test_cls)
ann_prob = ann_cls.predict_proba(X_test_cls)[:, 1]

ann_cls_metrics = pd.Series({
    "Accuracy": accuracy_score(y_test_cls, ann_pred),
    "Precision": precision_score(y_test_cls, ann_pred),
    "Recall": recall_score(y_test_cls, ann_pred),
    "F1": f1_score(y_test_cls, ann_pred),
    "ROC-AUC": roc_auc_score(y_test_cls, ann_prob),
    "Log Loss": log_loss(y_test_cls, ann_prob)
})
ann_cls_metrics.round(4)


## 15. ANN — Regression


In [ ]:
ann_reg = Pipeline([
    ("scaler", StandardScaler()),
    ("model", MLPRegressor(
        hidden_layer_sizes=(64, 32),
        activation="relu",
        solver="adam",
        max_iter=700,
        random_state=RANDOM_STATE
    ))
])

ann_reg.fit(X_train_reg, y_train_reg)
ann_reg_pred = ann_reg.predict(X_test_reg)

ann_reg_mse = mean_squared_error(y_test_reg, ann_reg_pred)
pd.Series({
    "MSE": ann_reg_mse,
    "RMSE": np.sqrt(ann_reg_mse),
    "MAE": mean_absolute_error(y_test_reg, ann_reg_pred),
    "R2": r2_score(y_test_reg, ann_reg_pred)
}).round(4)


## 16. CNN — Course Reference

CNNs are neural networks designed around convolution operations and are especially useful for image-like spatial data.

The following cell is **optional**. It uses the built-in `digits` dataset and runs only if TensorFlow is already installed. The rest of this notebook does **not** require TensorFlow.


In [ ]:
try:
    import tensorflow as tf
    from tensorflow import keras

    tf.random.set_seed(RANDOM_STATE)

    X_img = digits.images.astype("float32") / 16.0
    X_img = X_img[..., np.newaxis]
    y_img = digits.target

    X_train_img, X_test_img, y_train_img, y_test_img = train_test_split(
        X_img, y_img, test_size=0.2, stratify=y_img, random_state=RANDOM_STATE
    )

    cnn = keras.Sequential([
        keras.layers.Input(shape=(8, 8, 1)),
        keras.layers.Conv2D(8, (3, 3), activation="relu", padding="same"),
        keras.layers.MaxPooling2D((2, 2)),
        keras.layers.Flatten(),
        keras.layers.Dense(32, activation="relu"),
        keras.layers.Dense(10, activation="softmax")
    ])

    cnn.compile(
        optimizer="adam",
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    cnn.fit(X_train_img, y_train_img, epochs=3, batch_size=32, verbose=0)
    cnn_loss, cnn_acc = cnn.evaluate(X_test_img, y_test_img, verbose=0)

    print("CNN test accuracy:", round(float(cnn_acc), 4))
    print("CNN test loss:", round(float(cnn_loss), 4))

except ImportError:
    print("TensorFlow is not installed. CNN code is included as an optional course example.")


## 17. Course Coverage Summary

| Topic | Demonstrated here |
|---|---|
| Classification | ✅ |
| Logistic Regression | ✅ |
| Naive Bayes | ✅ |
| KNN | ✅ |
| SVM | ✅ |
| Decision Tree classification | ✅ |
| Linear Regression | ✅ |
| KNN regression | ✅ |
| SVM regression | ✅ |
| Decision Tree regression | ✅ |
| Missing-value imputation | ✅ |
| Encoding | ✅ |
| Scaling | ✅ |
| Train/test split | ✅ |
| Accuracy / Precision / Recall / F1 | ✅ |
| ROC-AUC | ✅ |
| Confusion Matrix | ✅ |
| Log Loss | ✅ |
| MSE / RMSE / MAE / R² | ✅ |
| 5-Fold Cross Validation | ✅ |
| Hyperparameter Tuning | ✅ |
| K-Means | ✅ |
| K-Means++ | ✅ |
| DBSCAN | ✅ |
| GMM | ✅ |
| PCA | ✅ |
| Reinforcement Learning | ✅ Q-Learning example |
| ANN classification | ✅ |
| ANN regression | ✅ |
| Activation functions | ✅ |
| Optimizers | ✅ explained + Adam used |
| CNN | ✅ optional TensorFlow example |

### Relation to the Student Risk project
The Student Risk project is a **supervised binary classification** problem. Therefore the official project uses classification methods appropriate to that task.  
This companion notebook exists to demonstrate the wider course syllabus without forcing unsuitable algorithms into the Student Risk model.


## 18. Viva Cheat Sheet

- **Classification:** predict a category/class.
- **Regression:** predict a continuous value.
- **Clustering:** discover groups without target labels.
- **PCA:** reduce dimensions while preserving as much variance as possible.
- **Cross-validation:** estimate model stability across multiple splits.
- **Hyperparameter tuning:** search for better model settings.
- **MSE:** regression error metric; lower is better.
- **Log Loss:** classification probability loss; lower is better.
- **ANN:** fully connected neural network.
- **CNN:** neural network using convolution, commonly for image/spatial data.
- **RL:** learn actions from rewards through interaction.
